<a href="https://colab.research.google.com/github/Houcineaberhache/Movie-Recommendation-Platform-ML-Advanced-Python-Project/blob/main/Data_prepocessing_pipline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Cell 1: install dependencies ──────────────────────────────────────────────
!pip install pandas numpy scipy scikit-learn requests tqdm --quiet

import pandas as pd
import numpy as np
import requests, zipfile, io, os
from tqdm import tqdm

print("All libraries loaded.")

All libraries loaded.


In [ ]:
# ── Cell 2: download MovieLens 25M ────────────────────────────────────────────
ML_URL = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"

print("Downloading MovieLens 25M (~250MB)...")
r = requests.get(ML_URL, stream=True)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall("data/")
print("Done. Files:", os.listdir("data/ml-25m/"))

Done. Files: ['movies.csv', 'genome-tags.csv', 'README.txt', 'ratings.csv', 'tags.csv', 'genome-scores.csv', 'links.csv']


In [ ]:
# ── Cell 3: download IMDb datasets ────────────────────────────────────────────
import urllib.request

IMDB_FILES = {
    "title.basics": "https://datasets.imdbws.com/title.basics.tsv.gz",
    "title.ratings": "https://datasets.imdbws.com/title.ratings.tsv.gz",
}

os.makedirs("data/imdb", exist_ok=True)
for name, url in IMDB_FILES.items():
    dest = f"data/imdb/{name}.tsv.gz"
    print(f"Downloading {name}...")
    urllib.request.urlretrieve(url, dest)

print("IMDb files ready.")

IMDb files ready.


In [ ]:
# ── Cell 4: TMDB API key setup ────────────────────────────────────────────────
# Register free at https://www.themoviedb.org/settings/api
TMDB_API_KEY = "91ec9f8e1b8ff540e6af31e73d7c603b"

def tmdb_get(movie_id):
    """Fetch metadata for one TMDB movie id."""
    url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    r = requests.get(url, params={"api_key": TMDB_API_KEY})
    return r.json() if r.status_code == 200 else None

# Quick test
test = tmdb_get(550)  # Fight Club
print(test.get("title"), "—", test.get("release_date"))

Fight Club — 1999-10-15


In [ ]:
# ── Cell 5: load MovieLens core files ─────────────────────────────────────────
ratings = pd.read_csv("/content/data/ml-25m/ratings.csv")
movies  = pd.read_csv("/content/data/ml-25m/movies.csv")
tags    = pd.read_csv("/content/data/ml-25m/tags.csv")
links   = pd.read_csv("/content/data/ml-25m/links.csv")   # bridges ML <-> TMDB/IMDb

print(ratings.shape, movies.shape, tags.shape, links.shape)
ratings.head()



(25000095, 4) (62423, 3) (1093360, 4) (62423, 3)


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [ ]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
tags.head()

,userId,movieId,tag,timestamp
0,3,260,classic,1439472355
1,3,260,sci-fi,1439472256
2,4,1732,dark comedy,1573943598
3,4,1732,great dialogue,1573943604
4,4,7569,so bad it's good,1573943455


In [ ]:
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [ ]:
# ── Cell 6: EDA snapshot ──────────────────────────────────────────────────────
print("=== RATINGS ===")
print(ratings.describe())
print("\nNull counts:\n", ratings.isnull().sum())
print("\nUnique users:", ratings["userId"].nunique())
print("Unique movies:", ratings["movieId"].nunique())

print("\n=== MOVIES ===")
print(movies.dtypes)
print(movies["genres"].value_counts().head(10))

=== RATINGS ===
             userId       movieId        rating     timestamp
count  2.500010e+07  2.500010e+07  2.500010e+07  2.500010e+07
mean   8.118928e+04  2.138798e+04  3.533854e+00  1.215601e+09
std    4.679172e+04  3.919886e+04  1.060744e+00  2.268758e+08
min    1.000000e+00  1.000000e+00  5.000000e-01  7.896520e+08
25%    4.051000e+04  1.196000e+03  3.000000e+00  1.011747e+09
50%    8.091400e+04  2.947000e+03  3.500000e+00  1.198868e+09
75%    1.215570e+05  8.623000e+03  4.000000e+00  1.447205e+09
max    1.625410e+05  2.091710e+05  5.000000e+00  1.574328e+09

Null counts:
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Unique users: 162541
Unique movies: 59047

=== MOVIES ===
movieId     int64
title      object
genres     object
dtype: object
genres
Drama                   9056
Comedy                  5674
(no genres listed)      5062
Documentary             4731
Comedy|Drama            2386
Drama|Romance           2126
Horror                  1661
C

In [ ]:
# ── Cell 7: rating distribution (text-based, no matplotlib needed) ────────────
dist = ratings["rating"].value_counts().sort_index()
for score, count in dist.items():
    bar = "☆" * int(count / 100000)
    print(f"{score:.1f} | {bar} ({count:,})")

0.5 | ☆☆☆ (393,068)
1.0 | ☆☆☆☆☆☆☆ (776,815)
1.5 | ☆☆☆ (399,490)
2.0 | ☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆ (1,640,868)
2.5 | ☆☆☆☆☆☆☆☆☆☆☆☆ (1,262,797)
3.0 | ☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆ (4,896,928)
3.5 | ☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆ (3,177,318)
4.0 | ☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆ (6,639,798)
4.5 | ☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆ (2,200,539)
5.0 | ☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆☆ (3,612,474)


In [ ]:
# ── Cell 8: clean ratings ─────────────────────────────────────────────────────
print("Before:", ratings.shape)

# Drop duplicates (same user rating same movie twice)
ratings = ratings.drop_duplicates(subset=["userId", "movieId"])

# Fix timestamp to datetime
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")

# Keep only users with >= 20 ratings (avoid cold-start noise)
user_counts = ratings["userId"].value_counts()
active_users = user_counts[user_counts >= 20].index
ratings = ratings[ratings["userId"].isin(active_users)]

# Keep only movies with >= 10 ratings
movie_counts = ratings["movieId"].value_counts()
active_movies = movie_counts[movie_counts >= 10].index
ratings = ratings[ratings["movieId"].isin(active_movies)]

print("After:", ratings.shape)

Before: (25000095, 4)
After: (24890583, 4)


In [ ]:
# ── Cell 9: clean movies dataframe ────────────────────────────────────────────
# Extract release year from title: "Toy Story (1995)" → 1995
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)$").astype(float)
movies["title_clean"] = movies["title"].str.replace(r"\s*\(\d{4}\)$", "", regex=True).str.strip()

# Split genres into a list
movies["genre_list"] = movies["genres"].str.split("|")
movies.loc[movies["genres"] == "(no genres listed)", "genre_list"] = None
movies = movies.dropna(subset=["genre_list"])

print(movies.head(3).to_string())

   movieId                    title                                       genres    year       title_clean                                         genre_list
0        1         Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy  1995.0         Toy Story  [Adventure, Animation, Children, Comedy, Fantasy]
1        2           Jumanji (1995)                   Adventure|Children|Fantasy  1995.0           Jumanji                     [Adventure, Children, Fantasy]
2        3  Grumpier Old Men (1995)                               Comedy|Romance  1995.0  Grumpier Old Men                                  [Comedy, Romance]


In [ ]:
# ── Cell 10: load IMDb basics and ratings ─────────────────────────────────────
imdb_basics  = pd.read_csv("/content/data/imdb/title.basics.tsv.gz", sep="\t",
                            na_values="\\N", low_memory=False)
imdb_ratings = pd.read_csv("/content/data/imdb/title.ratings.tsv.gz", sep="\t",
                            na_values="\\N")

# Keep only movies
imdb_basics = imdb_basics[imdb_basics["titleType"] == "movie"]
imdb_basics = imdb_basics[["tconst", "primaryTitle", "startYear",
                             "runtimeMinutes", "genres"]]
print(imdb_basics.shape)

(741907, 5)


In [ ]:
# ── Cell 11: merge MovieLens + IMDb via links ─────────────────────────────────
# links.csv has: movieId | imdbId | tmdbId
# IMDb tconst format is "tt0114709" → we need to zero-pad imdbId
links["tconst"] = "tt" + links["imdbId"].astype(str).str.zfill(7)

# Merge: movies → links → imdb_basics
df = movies.merge(links[["movieId", "tconst", "tmdbId"]], on="movieId", how="left")
df = df.merge(imdb_basics, on="tconst", how="left")
df = df.merge(imdb_ratings, on="tconst", how="left")  # IMDb score + votes

print("Merged shape:", df.shape)
df.head(3)

Merged shape: (57361, 14)


,movieId,title,genres_x,year,title_clean,genre_list,tconst,tmdbId,primaryTitle,startYear,runtimeMinutes,genres_y,averageRating,numVotes
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",tt0114709,862.0,Toy Story,1995.0,81,"Adventure,Animation,Comedy",8.3,1168517.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,"[Adventure, Children, Fantasy]",tt0113497,8844.0,Jumanji,1995.0,104,"Adventure,Comedy,Family",7.1,411225.0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,"[Comedy, Romance]",tt0113228,15602.0,Grumpier Old Men,1995.0,101,"Comedy,Romance",6.7,31598.0


In [ ]:
# ── Cell 12: enrich with TMDB API (batched, respects rate limit) ──────────────
import time

def fetch_tmdb_batch(tmdb_ids, delay=0.25):
    """Fetch overview + poster_path + vote_average from TMDB for a list of ids."""
    results = []
    for tmdb_id in tqdm(tmdb_ids):
        if pd.isna(tmdb_id):
            results.append({})
            continue
        data = tmdb_get(int(tmdb_id))
        if data:
            results.append({
                "tmdbId": tmdb_id,
                "overview": data.get("overview"),
                "poster_path": data.get("poster_path"),
                "tmdb_score": data.get("vote_average"),
                "tmdb_votes": data.get("vote_count"),
            })
        else:
            results.append({"tmdbId": tmdb_id})
        time.sleep(delay)   # stay within free-tier rate limit
    return pd.DataFrame(results)

# WARNING: full 25M dataset has ~60K movies — this takes time.
# For development, run on a sample first:
sample_ids = df["tmdbId"].dropna().unique()[:500]
tmdb_meta = fetch_tmdb_batch(sample_ids)

df = df.merge(tmdb_meta, on="tmdbId", how="left")
print("With TMDB enrichment:", df.shape)

100%|██████████| 500/500 [03:13<00:00,  2.59it/s]

With TMDB enrichment: (57361, 18)
